# SQL monitoring: one baseline, four comparisons

This executable example fits a small synthetic Tweedie burn-cost model, saves its monitoring baseline as JSON in SQL, and monitors a later dataset. The recurring steps use SQL and the new data after the publication files have been deleted.

The model has grouped regions, ordered bonus-malus levels with an explicit Unknown level, and a continuous age spline. Burn cost means loss per exposure; exposure is the fitting weight.

This notebook uses only an isolated SQLite database. One clearly labelled helper simulates the published/current deployment rows that a remote deployment would create. Ordinary local notebook APIs still produce audit packages and do not deploy models. No SQL Server connection or credentials are used.

| Variant | Coefficients | Smoothing lambdas | Data-driven knots |
|---|---|---|---|
| STATIC_SCORE | Baseline | Baseline | Baseline |
| FROZEN_REFIT | Refit | Fixed | Fixed |
| REESTIMATE_LAMBDA | Refit | Re-estimated | Fixed |
| FULL_ADAPTIVE | Refit | Re-estimated | Repositioned |

Run all cells using the project's Python environment. The first run writes to state/sql_monitoring_demo; subsequent runs create separate run directories beneath it. Existing 01/02 notebooks are not involved.


In [1]:
import json
import sys
from dataclasses import asdict
from pathlib import Path

import pandas as pd
from sqlalchemy import text
from superglm import Categorical, OrderedCategorical, Spline, SuperGLM, Tweedie, collapse_levels

from pricing_pipeline import notebook as pricing_api
from pricing_pipeline.data.manifest import (
    ModelFrameManifestSpec,
    create_model_frame_manifest_with_split,
)
from pricing_pipeline.modeling.monitoring import (
    MonitoringVariant,
    check_monitoring_data,
    persist_monitoring_fit,
    run_monitoring_fit,
)

PROJECT_ROOT = next(
    directory for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "pyproject.toml").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.demo_sql_monitoring import (
    create_demo_directory,
    export_sql_tables,
    simulate_demo_deployment,
    synthetic_burn_cost,
)

# Jupyter supplies display; this fallback also permits plain Python execution.
try:
    display
except NameError:
    display = print

OUTPUT = create_demo_directory(PROJECT_ROOT / "state" / "sql_monitoring_demo")
pricing = pricing_api.connect(mode="local", local_root=OUTPUT / "sqlite")
print(f"All demo output: {OUTPUT}")


All demo output: /home/max/projects/superglm-pricing-pipeline/.worktrees/sql-monitoring-baseline/state/sql_monitoring_demo/run-_p5hd98r


## 1. Declare and fit the baseline

The baseline contains 480 synthetic policies dated 2026-08-31. A compound Poisson-gamma generator supplies non-negative losses, including zeros. Both fit and export use exposure weights. The fit uses REML with two validation folds and retain_fit_state=False.


In [2]:
baseline_df = synthetic_burn_cost(rows=480, seed=1701, as_of="2026-08-31")
dataset = pricing_api.PricingDataset(
    baseline_df, name="synthetic_burn_cost", source="synthetic_demo",
    key="policy_id", as_of="as_of",
)

feature_names = ("region", "bonus_malus", "driver_age")
spec = pricing_api.PricingModelSpec(
    name="SQL_DEMO_BURN_COST",
    label="Synthetic burn cost",
    model_type="burn_cost",
    target="burn_cost",
    deployment_slot="DEMO_CURRENT",
    dataset=dataset,
    features=feature_names,
    sample_weight_column="exposure",
    export_weight_column="exposure",
    fit_mode="fit_reml",
    validation=pricing_api.ValidationSplitConfig.kfold(n_splits=2, random_state=19),
)

glm = SuperGLM(
    family=Tweedie(p=1.5),
    selection_penalty=0.0,
    retain_fit_state=False,
    features={
        "region": Categorical(
            base="North",
            grouping=collapse_levels(baseline_df.region, groups={"EastWest": ["East", "West"]}),
        ),
        "bonus_malus": OrderedCategorical(
            order=["0", "1", "2", "3", "4"],
            specials=["Unknown"],
            base="0",
            basis=Spline("cr", k=3, knot_strategy="quantile"),
        ),
        "driver_age": Spline("cr", k=3, knot_strategy="quantile"),
    },
)
registered = pricing_api.register_model(
    pricing, spec, source_root=PROJECT_ROOT / "tutorials" / "sql_monitoring",
    created_by="synthetic-demo",
)
candidate = pricing_api.fit_model(
    pricing, model=registered, frame=baseline_df,
    superglm_model=glm, created_by="synthetic-demo",
)
display(baseline_df.head(5))
display(pd.DataFrame([candidate.metrics]).round(4))


,policy_id,as_of,region,bonus_malus,driver_age,exposure,burn_cost
0,0,2026-08-31,South,0,21.383962,1.276383,1350.072511
1,1,2026-08-31,South,3,74.903695,0.806185,1464.469776
2,2,2026-08-31,East,0,26.887111,0.695058,0.000000
3,3,2026-08-31,North,4,42.363722,1.155516,0.000000
4,4,2026-08-31,West,0,24.524095,1.271834,407.928154


,cv_mean_deviance,cv_mean_nll,cv_mean_gini,cv_pooled_deviance,cv_pooled_nll,cv_std_deviance,cv_std_nll,cv_std_gini,cv_oof_coverage,fit_converged,fit_n_iter,fit_deviance,fit_effective_df,fit_phi,fit_log_likelihood,fit_null_log_likelihood,fit_null_deviance,fit_explained_deviance,fit_pearson_chi2,fit_n_obs,fit_likelihood_size,fit_reml_enabled,fit_reml_converged,fit_reml_n_iter
0,42.1141,5.3492,0.094,42.1141,5.3492,1.0361,0.0127,0.0427,1.0,1.0,1.0,19178.1841,6.7312,31.9715,-2550.5706,-2559.208,19730.4851,0.028,15131.1096,480.0,480.0,1.0,1.0,9.0


## 2. Save the model and inspect the SQL capture

Saving allocates the package, successful model run, and recipe revision. It also captures the immutable monitoring JSON in the same transaction. The local package initially has status LOCAL_AUDIT.

The baseline row stores fitted scoring state, refit settings, and aggregate reference profiles. Its source lineage points to the model run, package, recipe, receipt, and hashes. It contains no saved training dataframe or serialized model object.


In [3]:
saved = pricing_api.save_model_version(pricing, candidate)
with pricing.engine.connect() as connection:
    publication_rows = pd.read_sql_query(text("""
        SELECT mr.model_run_id, mr.model_version, mr.run_status,
               rp.rate_package_id, rp.package_status, recipe.recipe_revision,
               baseline.capture_status, baseline.snapshot_schema_version,
               LENGTH(baseline.snapshot_json) AS snapshot_characters,
               baseline.snapshot_sha256
        FROM pricing.MODEL_RUN AS mr
        JOIN pricing.PRICING_RATE_PACKAGE AS rp ON rp.rate_package_id=mr.rate_package_id
        JOIN pricing.MODEL_RECIPE AS recipe ON recipe.recipe_id=mr.recipe_id
        JOIN pricing.MODEL_MONITORING_BASELINE AS baseline ON baseline.model_run_id=mr.model_run_id
    """), connection)
display(publication_rows)
assert publication_rows.capture_status.tolist() == ["CAPTURED"]
assert saved.package_status == "LOCAL_AUDIT"


,model_run_id,model_version,run_status,rate_package_id,package_status,recipe_revision,capture_status,snapshot_schema_version,snapshot_characters,snapshot_sha256
0,1,v1,SUCCESS,1,LOCAL_AUDIT,1,CAPTURED,1,32070,ee7447fd6c01545967df4e1fefda1a65706c01b043887e05a745ecc485ea3421


## 3. Simulate the deployment, then remove the publication files

The following helper is only demo setup. It checks the SQLite dialect, the marked demo directory, and every attached database path before creating the one simulated deployment. It keeps all database guards enabled.

The next cell deletes only the three publication files created in this run. The monitoring steps below cannot load their saved joblib bundle or receipt file.


In [4]:
deployment_id = simulate_demo_deployment(
    pricing, saved, directory=OUTPUT, slot=registered.config.deployment_slot,
)
publication_files = [
    Path(candidate.completed_build.candidate_artifact_path),
    Path(candidate.completed_build.rating_workbook_path),
    Path(candidate.completed_build.publication_receipt_path),
]
for path in publication_files:
    assert path.resolve().is_relative_to(OUTPUT)
    path.unlink()
del candidate, glm, baseline_df, dataset, spec, registered
print(f"Simulated deployment: {deployment_id}")
print("Publication files remaining:", sum(path.exists() for path in publication_files))


Simulated deployment: 1
Publication files remaining: 0


## 4. Register a later dataset and load the baseline from SQL

The new snapshot contains 360 policies dated 2026-09-15. It has 12% claims inflation in the generating process and a larger North region share. Its actual outcomes remain random, so estimated changes need not equal the generator's inflation.

The manifest binds the exact new dataframe, date, target, and weight/export roles. It does not create another model version.

The saved snapshot requires the same SuperGLM version and a matching Python major/minor version. Loading checks that compatibility even though model files are no longer needed.


In [5]:
monitoring_df = synthetic_burn_cost(
    rows=360, seed=1702, as_of="2026-09-15", monitoring=True,
)
monitoring_manifest = create_model_frame_manifest_with_split(
    pricing.engine,
    frame=monitoring_df,
    spec=ModelFrameManifestSpec(
        dataset_name="synthetic_burn_cost",
        source_system="synthetic_demo",
        data_as_of_date="2026-09-15",
        data_as_of_column="as_of",
        pk_columns=("policy_id",),
        target_column="burn_cost",
        weight_column="exposure",
        export_weight_column="exposure",
        feature_columns=feature_names,
    ),
    validation_split=pricing_api.ValidationSplitConfig(
        method="none", n_splits=None, shuffle=False,
    ),
    created_by="synthetic-demo",
)
registered = pricing_api.load_registered_model(
    pricing, model_name="SQL_DEMO_BURN_COST", deployment_slot="DEMO_CURRENT",
    source_root=PROJECT_ROOT / "tutorials" / "sql_monitoring",
)
baseline = pricing_api.load_monitoring_baseline(pricing, model=registered)
X = monitoring_df.loc[:, list(feature_names)]
y = monitoring_df.burn_cost
weights = monitoring_df.exposure
display(pd.DataFrame([{
    "model_run_id": baseline.model_run_id,
    "deployment_id": baseline.deployment_id,
    "new_manifest_id": monitoring_manifest.manifest_id,
    "snapshot_sha256": baseline.snapshot_sha256,
    "reference_source": "SQL aggregate profiles",
}]))


,model_run_id,deployment_id,new_manifest_id,snapshot_sha256,reference_source
0,1,1,synthetic_burn_cost_20260915_63011cd21f,ee7447fd6c01545967df4e1fefda1a65706c01b043887e05a745ecc485ea3421,SQL aggregate profiles


## 5. Review support and drift before fitting

Preflight compares the new data with the stored categorical and ordered profiles. It also checks support for the fitted spline domains. Review warnings, then stop if there are compatibility errors.


In [6]:
preflight = check_monitoring_data(baseline, X, sample_weight=weights)
display(preflight.issues)
display(preflight.drift.round(4))
display(preflight.distributions.head(12))
preflight.raise_for_errors()
print(f"Compatible: {preflight.compatible}; reference: {preflight.reference_source}")


,feature,severity,code,message,affected_rows,affected_weight
0,driver_age,warning,SPLINE_RANGE_LOSS,"Spline feature 'driver_age' now has positive-weight range [20.0537, 77.9803] against saved domain [18.082, 79.917]. Parts of the saved curve have no new observations supporting them; review before interpreting changes there.",0,None
1,region,warning,CATEGORICAL_DRIFT,Feature 'region' has a large categorical mix change. Check upstream SQL definitions and portfolio composition; a new baseline may be needed. Drift alone does not identify the cause.,0,None


,feature,row_distance,weight_distance,threshold,needs_review
0,region,0.216,0.2140,0.2,True
1,bonus_malus,0.000,0.0117,0.2,False


,feature,level_key,level,reference_rows,current_rows,reference_row_share,current_row_share,reference_weight,current_weight,reference_weight_share,current_weight_share
0,region,"{""type"":""string"",""value"":""East""}",East,120,64,0.250000,0.177778,124.156590,65.000063,0.253646,0.184823
1,region,"{""type"":""string"",""value"":""North""}",North,103,155,0.214583,0.430556,107.736982,152.676795,0.220102,0.434127
2,region,"{""type"":""string"",""value"":""South""}",South,133,66,0.277083,0.183333,132.566818,63.729119,0.270828,0.181210
3,region,"{""type"":""string"",""value"":""West""}",West,124,75,0.258333,0.208333,125.027048,70.281308,0.255424,0.199840
4,bonus_malus,"{""type"":""string"",""value"":""0""}",0,80,60,0.166667,0.166667,82.812357,59.365692,0.169182,0.168802
5,bonus_malus,"{""type"":""string"",""value"":""1""}",1,80,60,0.166667,0.166667,81.140480,56.608606,0.165766,0.160963
6,bonus_malus,"{""type"":""string"",""value"":""2""}",2,80,60,0.166667,0.166667,78.693265,56.233956,0.160767,0.159898
7,bonus_malus,"{""type"":""string"",""value"":""3""}",3,80,60,0.166667,0.166667,80.197902,61.721220,0.163841,0.175500
8,bonus_malus,"{""type"":""string"",""value"":""4""}",4,80,60,0.166667,0.166667,83.368568,58.522439,0.170318,0.166405
9,bonus_malus,"{""type"":""string"",""value"":""Unknown""}",Unknown,80,60,0.166667,0.166667,83.274866,59.235372,0.170127,0.168432


Compatible: True; reference: sql_baseline


## 6. Run and persist all four comparisons

All four fits complete before persistence starts. Each observation has its own SQL transaction.

Each observation uses the same baseline and dated manifest. The fit contract freezes a common comparison grid. Persistence verifies the snapshot and deployment again, writes the evidence, and seals the observation. Repeating the same observation reuses its monitor_run_id.

These refits do not publish another package or replace the deployed baseline.


In [7]:
fit_results = {}
for variant in MonitoringVariant:
    result = run_monitoring_fit(
        baseline, X, y,
        variant=variant,
        sample_weight=weights,
        model_frame=monitoring_df,
        target_column="burn_cost",
        continuous_points=13,
        max_reml_iter=15,
    )
    fit_results[variant.value] = result

persisted_rows = []
for variant_name, result in fit_results.items():
    saved_observation = persist_monitoring_fit(
        pricing.engine, result,
        baseline_model_run_id=baseline.model_run_id,
        baseline_deployment_id=baseline.deployment_id,
        manifest_id=monitoring_manifest.manifest_id,
        component_role="OTHER",
        created_by="synthetic-demo",
    )
    retry = persist_monitoring_fit(
        pricing.engine, result,
        baseline_model_run_id=baseline.model_run_id,
        baseline_deployment_id=baseline.deployment_id,
        manifest_id=monitoring_manifest.manifest_id,
        component_role="OTHER",
        created_by="synthetic-demo",
    )
    assert retry.monitor_run_id == saved_observation.monitor_run_id
    assert retry.deduplicated
    persisted_rows.append({
        "variant": variant_name,
        **asdict(saved_observation),
        "identical_retry_reused": retry.deduplicated,
    })
display(pd.DataFrame(persisted_rows))


,variant,monitor_run_id,fit_contract_id,run_signature_sha256,deduplicated,identical_retry_reused
0,STATIC_SCORE,138f82c5-4567-4b78-8491-de69aaab0dec,d8723a4d-742e-4936-adee-b52c9d457e88,f30b174e8234cd0960546dd570cec1125924f66096cd724a7f073f8cf75ce502,False,True
1,FROZEN_REFIT,d9877b20-0fd2-4864-ba34-dc31ae8c2393,d8723a4d-742e-4936-adee-b52c9d457e88,2482e904740249914af7fe421727ebf94619bfc2c598ca24e44e78e1e37cef8d,False,True
2,REESTIMATE_LAMBDA,911d1e2e-0529-4ac6-8cd8-284f342912ca,d8723a4d-742e-4936-adee-b52c9d457e88,e97f598bdaa9558158a9558997d653ebbffebb6a4bfd76ba91bac0e210a98252,False,True
3,FULL_ADAPTIVE,f84c4638-45b2-4f33-b752-7af61ab9c19e,d8723a4d-742e-4936-adee-b52c9d457e88,c4f5385b041260aeaa84e09625c4673e70f56dd6f5daa6f19539cf61b8e2d592,False,True


## 7. Inspect the saved observations and relativities

The next tables read the persisted SQL rows. The categorical view compares like-for-like levels; continuous values use the same baseline grid. Lower deviance here describes fit to this monitoring snapshot and is not evidence of out-of-sample improvement.


In [8]:
with pricing.engine.connect() as connection:
    metric_rows = pd.read_sql_query(text("""
        SELECT run.variant_code AS variant, metric.metric_name, metric.metric_value
        FROM pricing.MODEL_MONITOR_RUN AS run
        JOIN pricing.MODEL_MONITOR_METRIC AS metric ON metric.monitor_run_id=run.monitor_run_id
    """), connection)
    relativity_rows = pd.read_sql_query(text("""
        SELECT run.variant_code AS variant, rel.term_name, rel.point_label,
               rel.point_numeric, rel.relativity
        FROM pricing.MODEL_MONITOR_RUN AS run
        JOIN pricing.MODEL_MONITOR_RELATIVITY AS rel ON rel.monitor_run_id=run.monitor_run_id
        ORDER BY rel.term_name, rel.point_label, rel.point_numeric, run.variant_code
    """), connection)
metrics = metric_rows.pivot(index="variant", columns="metric_name", values="metric_value")
display(metrics[[
    "row_count", "sample_weighted_mean_observed",
    "sample_weighted_mean_prediction", "deviance", "explained_deviance",
]].round(4))
region_comparison = relativity_rows.query("term_name == 'region'").pivot(
    index="point_label", columns="variant", values="relativity",
)
display(region_comparison.round(4))


metric_name,row_count,sample_weighted_mean_observed,sample_weighted_mean_prediction,deviance,explained_deviance
variant,,,,,
FROZEN_REFIT,360.0,419.6702,418.0809,15125.4820,0.0307
FULL_ADAPTIVE,360.0,419.6702,418.3801,15110.5814,0.0317
REESTIMATE_LAMBDA,360.0,419.6702,418.3757,15109.3857,0.0317
STATIC_SCORE,360.0,419.6702,376.3774,15407.1833,0.0127


variant,FROZEN_REFIT,FULL_ADAPTIVE,REESTIMATE_LAMBDA,STATIC_SCORE
point_label,,,,
East,0.8805,0.8848,0.8850,0.7097
North,1.0000,1.0000,1.0000,1.0000
South,1.1526,1.1553,1.1553,0.8030
West,0.8805,0.8848,0.8850,0.7097


## 8. Export the actual SQL tables

The Excel workbook contains a table guide, a readable model-feature sheet, and up to 20 actual rows from each relevant SQL table. The guide reports total row counts, exported row counts, sort order, purpose, and key links. Full JSON is saved alongside the workbook; long Excel cells carry a labelled preview.

SQL Server stores the monitoring observation and fit-contract tables in mlops. The SQLite mirror stores them in pricing so its views remain usable when its file is opened directly. MODEL_MONITORING_BASELINE is in pricing in both.


In [9]:
workbook_path, table_guide = export_sql_tables(pricing, directory=OUTPUT, limit=20)
display(table_guide[["SQL Server table", "total rows", "rows exported", "extract scope"]])
with pricing.engine.connect() as connection:
    assert connection.execute(text("SELECT COUNT(*) FROM pricing.MODEL_RUN")).scalar_one() == 1
    assert connection.execute(text("SELECT COUNT(*) FROM pricing.PRICING_RATE_PACKAGE")).scalar_one() == 1
    assert connection.execute(text("SELECT COUNT(*) FROM pricing.MODEL_MONITOR_RUN")).scalar_one() == 4
print(f"Excel workbook: {workbook_path}")
print(f"Full baseline JSON: {OUTPUT / 'baseline_snapshot.json'}")
print(f"SQLite database: {OUTPUT / 'sqlite' / 'pricing.sqlite'}")
pricing.engine.dispose()


,SQL Server table,total rows,rows exported,extract scope
0,pricing.PRICING_MODEL,1,1,All rows
1,pricing.MODEL_RECIPE,1,1,All rows
2,pricing.MODEL_RUN,1,1,All rows
3,pricing.PRICING_RATE_PACKAGE,1,1,All rows
4,pricing.MODEL_MONITORING_BASELINE,1,1,All rows
5,pricing.PRICING_MODEL_DEPLOYMENT,1,1,All rows
6,pricing.DATASET_MANIFEST,2,2,All rows
7,mlops.MODEL_FIT_CONTRACT,1,1,All rows
8,mlops.MODEL_MONITOR_RUN,4,4,All rows
9,mlops.MODEL_MONITOR_TERM,12,12,All rows


Excel workbook: /home/max/projects/superglm-pricing-pipeline/.worktrees/sql-monitoring-baseline/state/sql_monitoring_demo/run-_p5hd98r/sql_tables.xlsx
Full baseline JSON: /home/max/projects/superglm-pricing-pipeline/.worktrees/sql-monitoring-baseline/state/sql_monitoring_demo/run-_p5hd98r/baseline_snapshot.json
SQLite database: /home/max/projects/superglm-pricing-pipeline/.worktrees/sql-monitoring-baseline/state/sql_monitoring_demo/run-_p5hd98r/sqlite/pricing.sqlite


For an existing remote model, ordinary publication/deployment already supplies the SQL lineage. The recurring part starts at step 4 and uses the registered model's configured slot. Older published models without a snapshot can use capture_existing_monitoring_baseline once with a verified workbench Candidate; that migration reads the existing artifact and does not refit the model.
